# Getting started

This notebook walks through the core FactoriaX API: creating an environment, resetting it, stepping through episodes, and running batched rollouts on GPU with `jax.vmap`.

**Prerequisites**: install the package with `uv sync` (or `pip install factoriax`).

## Create an environment

`factoriax.make` mirrors `gymnax.make`: pass a registered scenario id and receive an `(env, params)` pair.
Available scenarios: `"EasyRocket-v1"` (16×16), `"Rocket-v1"` (32×32).

In [ ]:
import jax
import jax.numpy as jnp
import factoriax

env, params = factoriax.make("EasyRocket-v1")
print("observation space:", env.observation_space(params))
print("num actions:      ", factoriax.NUM_ITEM_TYPES)

## Reset and inspect state

In [ ]:
key = jax.random.PRNGKey(0)
key, reset_key = jax.random.split(key)

obs, state = env.reset_env(reset_key, params)
print("obs shape:", obs.shape)
print("step:     ", state.time)

## Step with a random action

In [ ]:
from factoriax import Action

key, step_key, action_key = jax.random.split(key, 3)
action = jax.random.randint(action_key, shape=(), minval=0, maxval=len(Action))

obs, state, reward, done, info = env.step_env(step_key, state, action, params)
print("reward:", reward)
print("done:  ", done)
print("step:  ", state.time)

## Run a short episode

Loop for 50 steps and collect rewards.

In [ ]:
key, reset_key = jax.random.split(key)
obs, state = env.reset_env(reset_key, params)

rewards = []
for _ in range(50):
    key, step_key, action_key = jax.random.split(key, 3)
    action = jax.random.randint(action_key, shape=(), minval=0, maxval=len(Action))
    obs, state, reward, done, info = env.step_env(step_key, state, action, params)
    rewards.append(float(reward))

print(f"total reward over 50 steps: {sum(rewards):.3f}")

## Batched rollouts with `jax.vmap`

FactoriaX state is pure JAX arrays, so `vmap` over a batch of environments runs the full batch in a single XLA kernel — no Python loop.

In [ ]:
N_ENVS = 64

vmap_reset = jax.vmap(env.reset_env, in_axes=(0, None))
vmap_step  = jax.vmap(env.step_env,  in_axes=(0, 0, 0, None))

keys = jax.random.split(jax.random.PRNGKey(1), N_ENVS)
obs_batch, state_batch = vmap_reset(keys, params)
print("batched obs shape:", obs_batch.shape)  # (64, obs_dim)

In [ ]:
# One batched step
step_keys   = jax.random.split(jax.random.PRNGKey(2), N_ENVS)
action_keys = jax.random.split(jax.random.PRNGKey(3), N_ENVS)
actions = jax.vmap(lambda k: jax.random.randint(k, shape=(), minval=0, maxval=len(Action)))(action_keys)

obs_batch, state_batch, rewards, dones, _ = vmap_step(step_keys, state_batch, actions, params)
print("rewards shape:", rewards.shape)   # (64,)
print("mean reward:  ", rewards.mean())

## Next steps

- {doc}`../guides/action-design` — how the compound action space is structured.
- {doc}`../api/index` — full API reference.
- `baselines/easy_rocket/ppo/` — PPO training loop using `AutoResetWrapper` and `jax.lax.scan`.